In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
source = pd.read_csv('/content/drive/MyDrive/파일위치/pan_qna_kkm_sbert.csv')
source.head()

,question,answer,embedding
0,개인 금융 기관 자금 대출 걸 나 대출금 상환 일련 사법 상 계약 아야,"행정심판법 제2조 제1항 및 제3조 제1항의 규정에 의하면, 행정심판은 행정청이 행...",[ 1.02470636e-01 -1.75006568e-01 2.76431054e-...
1,종전 규정 허가 자 신 규정 적용 매립 목적 변경 대상 알 니 라는 이유 행하 부당,기타 주변여건의 변화 등으로 인하여 매립목적의 변경이 불가피한 경우인지 등을 종합적...,[-1.66785493e-01 1.21014211e-02 8.48394454e-...
2,군 복무 중 질병 발병 악화 공상 판정 전역 이에 위원 회의 심의 도 군복무 중 걸...,청구인이 군 복무중 이 건 질병들이 발병 또는 악화되어 육군에서 공상판정을 받고 전...,[ 3.43711935e-02 2.06315592e-01 5.93770921e-...
3,토지 임차인 지상 물 매수 청구권 배제 기로 하여 임차인 불리 약정 효력 다,임차인이 임대차계약을 체결할 당시 건물 기타 지상시설 일체를 포기하기로 하였다 해도...,[-4.27980796e-02 1.16126850e-01 5.36623657e-...
4,낙뢰 위험 상당 정도 예상 체육 시설 업자 골프장 운영자 이용자 피난 지시 내리 주...,체육시설업자로서는 낙뢰의 위험이 상당한 정도로 예상되는 경우에는 이용자에 대하여 피...,[-2.64974952e-01 7.22772814e-03 9.41194773e-...


In [ ]:
def str_to_vector(emd):
    res = emd.split(' ')
    #print(emd)
    #print(len(r))
    r = []
    for a in res:
        if len(a) >2:
            r.append(a)
    if len(r) != 768:
        print(len(r))
    for i in range(len(r)):
        r[i] = r[i].replace('[', '')
        r[i] = r[i].replace(']', '')
        r[i] = r[i].replace('\n', '')
        r[i] = float(r[i])
    return np.array(r)

In [ ]:
import numpy as np
emd2 = source.apply(lambda x: str_to_vector(x['embedding']), axis=1)
answer = source['answer']
question = source['question']

In [ ]:
!pip install konlpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.4/19.4 MB 78.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 494.1/494.1 kB 30.5 MB/s eta 0:00:00


In [ ]:
from konlpy.tag import Kkma

kkma = Kkma()

In [ ]:
from tqdm import tqdm
import re
stopwords = ['하', '의', '에', '이', '는', '을', '가', '되', '를', '있', '여', '으로', '경우', '는가', '수', '아', '로', '그', '은', '고', '제', '것', '지', '에서', '저', '대하', '조', '어', '등', '행위', '관하', '해당', '보', '항', '에게', '및', '과', '받', '와', '없', '의하', '다고', '처분', '않', '또는', '였', '었', '나요', '따르']


def tokenize_sentence(sentence):
    tokenized_sentence = []
    tokenized_sentence = kkma.morphs(sentence) # 토큰화
    stopwords_removed_sentence = [word for word in tokenized_sentence if not word in stopwords] # 불용어 제거
    l = ''
    for s in stopwords_removed_sentence:
        s = re.sub(r"[^가-힣\s]", " ", s)
        l += s +' '
    return l

In [ ]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('sentence-transformers/xlm-r-100langs-bert-base-nli-stsb-mean-tokens')

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/4.09k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/527 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

1_Pooling%2Fconfig.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
from numpy import dot
from numpy.linalg import norm
def cos_sim(A, B):
  return dot(A, B)/(norm(A)*norm(B))

In [ ]:
from math import *

def jaccard_similarity(x, y):
    intersection_cardinality = set(x).intersection(set(y))
    union_cardinality = set(x).union(set(y))

    return len(intersection_cardinality) / len(union_cardinality)

In [ ]:
def return_answer(q):
    q = tokenize_sentence(q)
    embedding = model.encode(q)
    #유사도 계산 및 정렬
    sim_scores = [[k, 0.1*jaccard_similarity(question[k], q)+0.9*cos_sim(emd2[k], embedding)] for k in range(len(source))]
    sim_scores.sort(key=lambda x: x[1], reverse=True) #sim_scores의 각 리스트 중 두번째 요소를 정렬 기준으로

    #상위 다섯 개 저장(중복 제거)
    result = [] #전체
    score = []
    for sim in sim_scores:
        a = answer[sim[0]]
        if a not in result:
            result.append(a)
            score.append(sim[1])
        else:
            #print(a)
            pass
        if len(result) == 5:
            break
    print(score)
    return result

In [ ]:
return_answer('사실로 인한 명예훼손죄가 성립되는지?')

[0.8183368555488779, 0.8173255498620453, 0.8126182194389149, 0.8114739861093185, 0.8092103968291467]


['범죄사실을 부인하거나 죄의 뉘우침이 없는 자수는 그 외형은 자수일지라도 법률상 형의 감경사유가 되는 진정한 자수라고는 할 수 없다.',
 '형법 제307조는 명예훼손죄를 규정하며, 제1항은 사실의 적시로 명예를 훼손한 경우, 제2항은 허위 사실의 적시로 명예를 훼손한 경우를 다룹니다. 제1항은 진실, 허위 여부와 무관하게 적용되며, 허위 사실이나 행위자가 그 허위성을 인식하지 못한 경우에도 제1항이 적용됩니다.',
 '법인의 목적사업 수행에 영향을 미칠 정도로 법인의 사회적 명성, 신용을 훼손하여 법인의 사회적 평가가 침해된 경우에는 그 법인에 대하여 불법행위를 구성한다고 할 것이다.',
 '사기죄의 요건인 기망은 재산상 거래관계에 있어서 신의칙상 의무를 저버리는 적극적·소극적 행위를 뜻하며, 상대방을 착오에 빠지게 하여 재산적 처분행위를 하도록 하면 충분하다.',
 '위해를 가할 듯이 위협하여 손목시계 등을 교부받아 갈취한 공갈행위는사회보호법 제6조 제2항소정의 “동종 또는 유사한 죄”라 볼 수 없다.']

In [ ]:
return_answer('건물 소유자의 배우자와 한 임대차 계약의 효력')

[0.7713995668788729, 0.7462614194488738, 0.7347872567723085, 0.7343375713606487, 0.732874852383615]


['주택임대차보호법이 적용되는 임대차가 임차인과 주택의 소유자인 임대인 사이에 임대차계약이 체결된 경우로 한정되는 것은 아닙니다.',
 '그렇습니다. 건물 소유자가 현실적으로 건물을 점거하고 있지 않은 경우에도 건물의 소유자는 그 건물의 소유를 위하여 그 부지를 점유하고 있다고 인정할 수 있습니다.',
 '원심이 위에 판시한 바와 같이 위 망 소외 1이 이 사건 대지를 점유한 일이 없으며 원고도 아무리 빨라야 1984. 4. 20.경 비로소 이 사건 대지를 점유하였다고 할 것이므로 점유기간이 20년에 미달된다 하여 원고의 시효취득 주장을 배척한 것은 필요한 심리를 다하지 아니하였거나 채증법칙에 위배하여 사실을 오인하였거나 점유에 관한 법리를 오해함으로써 판결에 영향을 미친 위법을 범하였다는 비난을 면하기 어렵다고 할 것이다.',
 '임대주택법 등 관계 법령에 위반하여 우선매각 대상자가 아닌 제3자에게 이를 매각하였다는 사정만으로는 그 사법상의 효력이 무효로 되는 것은 아니다.',
 '원심판결에는 반소의 청구취지를 오해한 위법이 있다 할 것이나, 앞서 본 바와 같이 위 증축 부분은 원고의 소유이므로 어차피 반소청구는 기각될 것인바, 그렇다면 이는 상고인에게 더욱 불리한 결과가 되므로 당원으로서는 원심판결 중 반소 부분을 파기한 후 반소청구를 기각하는 대신 상고를 기각하기로 한다.']

In [ ]:
return_answer('녹음 증언의 효력')

[0.8120130266414524, 0.7164574112770487, 0.7026100690116367, 0.7005859392846975, 0.6829139740320687]


['그 대화내용을 녹음한 원본이거나 혹은 원본으로부터 복사한 사본일 경우에는 복사과정에서 편집되는 등의 인위적 개작 없이 원본의 내용 그대로 복사된 사본임이 입증되어야만 하고, 그러한 입증이 없는 경우에는 쉽게 그 증거능력을 인정할 수 없다.',
 '아니요. 구수증서에 의한 유언이 허용되는 급박한 사유가 있는 때가 아니므로 구수증서에 의한 유언을 할 수 없습니다.',
 '녹음테이프는 성질상 작성자나 진술자의 서명이나 날인이 없고, 내용이 편집·조작될 위험이 있으므로 그 대화내용을 녹음한 원본이거나 혹은 인위적 개작 없이 원본의 내용 그대로 복사된 사본임이 증명되어야만 하고, 녹음테이프에 수록된 대화내용과 녹취록의 기재가 일치하더라도 위와 같은 증명이 있다고 할 수 없다.',
 '공범관계의 공모는 공범자 간에 직·간접적으로 공동실행에 관한 암묵적 의사연락이 있으면 족하고, 정황사실과 경험법칙에 의하여 인정될 수 있다.',
 '유언자가 유언의 취지, 그 성명과 연월일을 구술하고 이에 참여한 증인이 유언의 정확함과 그 성명을 구술해야 합니다.']

In [ ]:
df = pd.read_csv('/content/drive/MyDrive/파일위치/pan_qna.csv')
n =10
sample = df.sample(n)
for i in sample.index:
  print(sample['question'][i])
  print(sample['answer'][i], '\n')

저명한 상품표지가 타인에 의하여 영업표지로 사용되는 경우에도 ‘식별력의 손상’이 생기는가?
‘식별력의 손상’은 ‘특정한 표지가 상품표지나 영업표지로서의 출처표시 기능이 손상되는 것’을 의미하는 것으로 해석함이 상당하며, 이러한 식별력의 손상은 저명한 상품표지가 다른 사람에 의하여 영업표지로 사용되는 경우에도 생긴다. 

게임물 관련 사업자가 제공 또는 승인하지 아니한 게임물을 제작, 배급, 제공 또는 알선해도 되는가?
인터넷 온라인 게임 ‘C’는 게임물 관련사업자인 (주)D가 제작, 배급, 제공하는 게임물 이고, 누구든지 게임물 관련사업자가 제공 또는 승인하지 아니한 게임물을 제작, 배급, 제공 또는 알선하는 행위를 하여서는 아니 된다. 

교차로에 횡형삼색등 신호기가 설치되어 있으나 달리 비보호좌회전 표지가 없는 경우, 녹색 등화시 유턴하여 진행하였다면 같은 진행방향의 후방차량에 대하여도 신호위반의 책임을 지나요?
같은 진행방향의 후방차량에 대하여도 책임을 지게 됩니다. 

이사회 내 위원회 위원는 음성 송수신 장치에 의해서도 결의참가가 가능하나요?
네. 이사회 내 위원회 위원는 음성 송수신 장치에 의해서도 결의참가가 가능합니다. 

공무수행사인의 경우 청탁금지법 제10조의 외부강의등 사례금 수수 제한 규정이 적용되나요?
청탁금지법의 적용대상이 아닙니다. 

부동산에 대한 강제집행정지 신청사건에서 담보제공명령을 받은 당사자가 아닌 제3자가 당사자를 대신하여 담보를 공탁한다는 취지를 공탁서에 기재함으로써 당사자를 위한 담보를 제공하였다면 제3자의 담보제공행위는 유효한가?
부동산에 대한 강제집행정지 신청사건에서 반드시 담보제공명령을 받은 당사자만이 담보를 제공하여야 한다고 볼 근거는 없으므로, 제3자 역시 당사자를 위한 담보를 제공할 수 있다. 

학교생활기록부 용어정의에서 사용자의 의미는 무엇인가요?
질의하신 사안의 경우, 학교에서의 ‘사용자'라 함은 실제로 학교생활기록부의 전산 업무를 총괄하도록 학교장의 임명을 받은 교사와 학생의 학교생활기록부를 직접 전산으

In [ ]:
return_answer('공무수행사도 청탁금지법의 대상이 되는지?')

[0.8380726212357431, 0.8336703179645535, 0.833554625921313, 0.8223933476459205, 0.8186161666760513]


['네 청탁금지법의 적용대상입니다.',
 '청탁금지법에서 열거한 14가지 청탁만 규율대상에 해당합니다.',
 '행정기관 계약직 직원의 경우 공무원 신분이 아니라면 공직자등에 해당하지 않는 반면, 공직유관단체 계약직 직원은 당해 기관과 직접 근로계약을 체결했다면 청탁금지법상 공직자등에 해당합니다.',
 '공직자등이 아닌 민간인에게 금품등을 제공하는 행위는 청탁금지법상 규율대상이 아닙니다.',
 '공무원, 공직유관단체 또는 공공기관의 임직원, 각급 학교의 장과 교직원, 언론사 대표자 및 임직원 등이 공직자등에 해당합니다.']

In [ ]:
return_answer('식별력의 손상의 의미')

[0.7109068405730339, 0.7109068405730339, 0.6920408623487224, 0.686521848908042, 0.6813255664336998]


['그 기재내용에 의하여 그 의사표시의 존재 및 내용을 인정하여야 할 것인데도 처분문서인 각 차용증의 기재내용과 무관하거나 신빙성 없는 증언들에 의하여 위의 처분문서의 증명력을 부인함으로써 사실을 그릇 인정한 위법이 있다.',
 '신빙성을 인정하기 어렵거나 증거로 삼을 수 없는 것들 또는 신빙성을 인정한다 하더라도 처분문서인 매매계약서나 차용증서의 증명력을 배척하기에는 부족한 증거들에 의하여 위의 처분문서의 증명력을 부인함으로써 채증법칙에 위배하여 처분문서의 증명력에 관한 법리를 오해하고 의사표시의 해석을 잘못한 위법을 법하였다.',
 '그 추상이 장래의 취직, 직종선택, 승진, 전직에의 가능성 등에 영향을 미칠 정도로 현저한 경우에는 그로 인한 노동능력상실이 없다 할 수는 없으므로 그 경우에는 추상장애로 인하여 노동능력상실이 있다고 보는 것이 상당하다.',
 '원심의 위와 같은 판단은 정당하고, 거기에 논하는 바와 같은 상표법 제6조 제1항 제3호에 관한 법리오해나 심리미진, 판단유탈의 위법이 있다고 할 수 없다. 본원상표는 이른바 조어상표(造語商標)에 해당한다고 할 수 있으나, 조어상표의 경우에도 그것이 지정상품의 성질을 표시하는 것으로 직감될 수 있는 한 기술적 상표에 해당될 수 있다고 할 것이므로 조어상표라는 이유만으로 반드시 그 상표가 식별력이 있다고 할 수는 없고(당원 1994. 8. 26. 선고 93후1100 판결 참조), 또한 논지는 본원상표가 세계 20여 개 국에서 등록이 된 점에 비추어 식별력이 있다는 취지이나, 출원상표의 등록 허부는 우리 상표법에 의하여 거래 기타 일반사회의 실정 등을 참작하여 독자적으로 결정하여야 할 것이므로 우리나라와 법제 기타의 사정이 같다고 할 수 없는 외국에서 본원상표가 등록되었다는 사유만으로 우리나라에서도 반드시 그 등록이 허용되어져야 하는 것은 아니라고 할 것이다(당원 1995. 3. 14. 선고 94후1701 판결 참조). 논지는 모두 이유가 없다.',
 '피고인은 2020. 8. 10.경부터 2020. 9.

In [ ]:
return_answer('비보호 좌회전 표시가 없는 교차로에서 초록불이 켜졌을 때 유턴하였다면 뒤쪽 차량에 대해 책임을 지는지 ')

[0.8257910682230754, 0.8257095563618172, 0.7990221396616588, 0.7936984333019458, 0.7891313997174699]


['같은 진행방향의 후방차량에 대하여도 책임을 지게 됩니다.',
 '신호위반의 책임은 인정되지 않습니다.',
 '교차로에 녹색, 황색 및 적색의 삼색등화만이 나오는 신호기가 설치되어 있고 달리 비보호좌회전 표시나 유턴을 허용하는 표시가 없는 경우에 차마의 좌회전 또는 유턴은 원칙적으로 허용되지 않는다고 보아야 한다.',
 '원칙적으로 허용되지 않습니다.',
 '횡형삼색등신호기가 교차로의 대각선 지점에 있지 아니하고 교차로에 연이어 있는 횡단보도상에 보행자 신호기와 함께 설치되어 있을 경우 이는 횡단보도상을 통행하는 보행자를 보호하기 의하여 차량들에 대한 횡단보도에 진입 또는 정지를 지시하는 신호기로 보아야 한다.']